# Registrar los productos MEDI (INEGI) en el catálogo STAC (`data/stac/inegi/`)

Paso **S4** de `docs/planes/03-capas-inegi-medi.md`. Abre el catálogo `atlas` existente
(creado por la libreta 003), reemplaza la colección `inegi` si ya estaba y
registra cuatro items con los productos de las libretas 005, 006, 008, 009 y
010. Indicadores: los siete componentes del MEDI y el índice, cada uno con
la escala en la que se midió (`scale_native`) para que la app avise cuando
un valor es heredado (p. ej. combustible municipal pintado por AGEB).

| item | asset `data` | qué es |
|---|---|---|
| `medi-ent-2020`  | `medi_ent_2020.parquet`  | GeoParquet, 32 estados (los 7 componentes nativos) |
| `medi-mun-2020`  | `medi_mun_2020.parquet`  | GeoParquet, 2 469 municipios |
| `medi-ageb-2020` | `medi_ageb_2020.parquet` | GeoParquet, 64 313 AGEB |
| `medi-grid-2020` | `medi_ageb_2020_grid.parquet` | tabla puente AGEB → celda UTCI |

Cada item usa la extensión **table** (columnas con tipo y descripción) y una
propiedad propia `atlas:indicators` con los indicadores que la app debe
ofrecer (id, etiqueta, columna, peso MEDI, numerador y denominador). La app
descubre los años y los indicadores desde aquí, nunca con rutas sueltas.
El STAC **cataloga, no duplica**: los assets apuntan por ruta relativa a
`data/derived/`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import pystac
import geopandas as gpd

ROOT = Path("..").resolve()
STAC_DIR = ROOT / "data" / "stac"
DERIVED = ROOT / "data" / "derived" / "INEGI" / "2020"

YEAR = 2020
PARQUET_MEDIA_TYPE = "application/vnd.apache.parquet"
TABLE_EXT = "https://stac-extensions.github.io/table/v1.2.0/schema.json"
LICENSE_URL = "https://www.inegi.org.mx/inegi/terminos.html"

PRODUCTOS = {
    "ent": DERIVED / "medi_ent_2020.parquet",
    "mun": DERIVED / "medi_mun_2020.parquet",
    "ageb": DERIVED / "medi_ageb_2020.parquet",
    "grid": DERIVED / "medi_ageb_2020_grid.parquet",
}
for p in PRODUCTOS.values():
    assert p.exists(), f"falta {p}: corre 005, 006, 008, 009 y 010"

# Indicadores que expone la app (orden y pesos del diseño MEDI). Campos:
#   column: porcentaje (o índice) a pintar; flag_column: ok|censurado|sin_viviendas;
#   upper_bound_column: cota superior si censurado; numerator/denominator: para
#   agregar por celda UTCI (si numerator es None se promedia `column` ponderado
#   por `denominator`); cv_column/quality_column: precisión muestral (ampliado);
#   scale_native: escala en que se midió, por nivel; unit: "%" o "pts";
#   is_index: True para el MEDI; components: pesos del índice.
def _ind(id, label, column, weight, **kw):
    base = {"id": id, "label": label, "column": column, "weight_medi": weight, "unit": "%",
            "flag_column": None, "upper_bound_column": None, "numerator": None, "denominator": "vivparh_cv",
            "cv_column": None, "quality_column": None, "is_index": False, "components": None,
            "scale_native": {"ent": "estado", "mun": "municipio", "ageb": "ageb"}}
    base.update(kw)
    return base


AMPLIADO_SCALE = {"ent": "estado", "mun": "municipio", "ageb": "localidad ≥ 50 k o municipio (heredado)"}
INDICATORS = [
    _ind("sin_elec", "Viviendas sin electricidad (%)", "p_sin_elec", 0.24,
         flag_column="flag_elec", upper_bound_column="p_sin_elec_sup", numerator="vph_s_elec"),
    _ind("sin_refri", "Viviendas sin refrigerador (%)", "p_sin_refri", 0.21,
         flag_column="flag_refri", numerator="vph_sin_refri"),
    _ind("sin_confort", "Sin confort térmico condicional al clima (%)", "p_sin_confort", 0.14,
         scale_native={"ent": "AC: estado (ampliado); calefacción: estado (ENCEVI 2018); clima: UTCI",
                       "mun": "AC: municipio (ampliado); calefacción: estado (ENCEVI 2018); clima: UTCI",
                       "ageb": "AC: localidad/municipio (ampliado, heredado); calefacción: estado (ENCEVI 2018); clima: celda UTCI"}),
    _ind("lena", "Cocinan con leña, carbón u otro combustible (%)", "p_lena", 0.13,
         cv_column="cv_lena", quality_column="calidad_lena", scale_native=AMPLIADO_SCALE),
    _ind("fogon_sin_chim", "Cocinan con leña o carbón sin chimenea (%)", "p_fogon_sin_chim", 0.13,
         cv_column="cv_fogon_sin_chim", quality_column="calidad_fogon_sin_chim", scale_native=AMPLIADO_SCALE),
    _ind("sin_telef", "Viviendas sin teléfono fijo ni celular (%)", "p_sin_telef", 0.08,
         flag_column="flag_telef", numerator="vph_sinltc"),
    _ind("sin_rtv", "Viviendas sin radio ni televisor (%)", "p_sin_rtv", 0.07,
         flag_column="flag_rtv", numerator="vph_sinrtv"),
    _ind("sin_ac", "Viviendas sin aire acondicionado (%) — insumo del confort térmico", "p_sin_ac", 0.0,
         cv_column="cv_sin_ac", quality_column="calidad_sin_ac", scale_native=AMPLIADO_SCALE),
    _ind("medi", "Índice MEDI (0–100)", "medi", 1.0, unit="pts", is_index=True,
         components={"sin_elec": 0.24, "sin_refri": 0.21, "sin_confort": 0.14, "lena": 0.13,
                     "fogon_sin_chim": 0.13, "sin_telef": 0.08, "sin_rtv": 0.07},
         scale_native={"ent": "7 componentes nativos del estado",
                       "mun": "4 componentes municipales (ITER) + 3 del ampliado municipal; calefacción estatal",
                       "ageb": "4 componentes propios de la AGEB + 3 heredados (localidad/municipio); calefacción estatal"}),
]


def indicators_for(level: str) -> list[dict]:
    """Copia de INDICATORS con `scale_native` resuelto para el nivel."""
    out = []
    for ind in INDICATORS:
        d = dict(ind)
        d["scale_native"] = ind["scale_native"][level]
        out.append(d)
    return out


# Descripciones de columnas para la extensión table (las que no estén aquí
# se registran sólo con nombre y tipo).
DESCRIPCIONES = {
    "cvegeo": "Clave geoestadística INEGI (13 = AGEB urbana, 5 = municipio)",
    "cve_ent": "Clave de entidad", "cve_mun": "Clave de municipio", "cve_loc": "Clave de localidad",
    "cve_ageb": "Clave de AGEB", "nom_ent": "Entidad", "nom_mun": "Municipio", "nom_loc": "Localidad",
    "ambito_mgn": "Capa del marco de donde vino el polígono (urbana | rural)",
    "pobtot": "Población total",
    "vivpar_hab": "Viviendas particulares habitadas",
    "vivparh_cv": "Viviendas particulares habitadas con características captadas (denominador de los VPH_*)",
    "vph_c_elec": "Viviendas con energía eléctrica", "vph_s_elec": "Viviendas sin energía eléctrica",
    "vph_refri": "Viviendas con refrigerador", "vph_sin_refri": "Viviendas sin refrigerador (vivparh_cv − vph_refri)",
    "vph_sinrtv": "Viviendas sin radio ni televisor", "vph_sinltc": "Viviendas sin línea telefónica ni celular",
    "p_sin_elec": "% viviendas sin electricidad (nulo si censurado o sin viviendas)",
    "p_sin_elec_sup": "Cota superior de p_sin_elec cuando el numerador está censurado (< 3 viviendas)",
    "p_sin_refri": "% viviendas sin refrigerador",
    "p_sin_telef": "% viviendas sin línea telefónica fija ni celular",
    "p_sin_rtv": "% viviendas sin radio ni televisor",
    "flag_elec": "ok | censurado | sin_viviendas", "flag_refri": "ok | censurado | sin_viviendas",
    "flag_telef": "ok | censurado | sin_viviendas", "flag_rtv": "ok | censurado | sin_viviendas",
    "p_lena": "% viviendas que cocinan con leña, carbón u otro combustible (≠ gas/electricidad), sobre las que cocinan; Cuestionario ampliado 2020",
    "p_fogon_sin_chim": "% viviendas que cocinan con leña/carbón sin chimenea, sobre las que cocinan; ampliado 2020",
    "p_sin_ac": "% viviendas sin aire acondicionado; ampliado 2020",
    "cv_lena": "Coeficiente de variación (%) de p_lena", "cv_fogon_sin_chim": "CV (%) de p_fogon_sin_chim", "cv_sin_ac": "CV (%) de p_sin_ac",
    "calidad_lena": "ok (CV≤15) | aviso (≤30) | poco_preciso", "calidad_fogon_sin_chim": "ok | aviso | poco_preciso", "calidad_sin_ac": "ok | aviso | poco_preciso",
    "n_muestra": "Viviendas en la muestra del ampliado (dominio de estimación)",
    "origen_ampliado": "localidad (≥ 50 k hab.) | municipio: de dónde heredó la AGEB los valores del ampliado",
    "h_calor": "Horas/año en niveles UTCI ≥ calor fuerte (promedio de los años del STAC)", "h_frio": "Horas/año en niveles UTCI ≤ frío moderado (< 0 °C)",
    "necesita_ac": "Regla climática: h_calor ≥ H_CALOR", "necesita_calef": "Regla climática: h_frio ≥ H_FRIO",
    "p_sin_calef": "% viviendas sin calefacción (ENCEVI 2018, estado)", "p_sin_calef_ent": "% sin calefacción, estado (ENCEVI 2018)",
    "p_sin_ac_encevi_ent": "% sin AC según ENCEVI 2018 (contraste)", "region_encevi": "Región climática ENCEVI (1 cálida extrema, 2 templada, 3 tropical)",
    "region_nombre": "Nombre de la región ENCEVI",
    "p_sin_confort": "% carencia de confort térmico condicional al clima (AC donde hace calor, calefacción donde hace frío)",
    "medi": "Índice MEDI 0–100 (suma ponderada de las 7 carencias)", "medi_min": "MEDI con los componentes censurados en 0",
    "medi_max": "MEDI con los componentes censurados en su cota superior", "n_censurados": "Componentes censurados (INEGI *) en la unidad",
    "n_ageb": "AGEB urbanas del municipio en el ITER", "pob_ageb_urb": "Población en AGEB urbanas",
    "cent_lon": "Longitud del centroide (calculado en EPSG:6372)", "cent_lat": "Latitud del centroide",
    "lat_c": "Latitud del centro de la celda UTCI de 0.25° que contiene el centroide",
    "lon_c": "Longitud del centro de la celda UTCI",
    "geometry": "Polígono del Marco Geoestadístico 2020, EPSG:4326",
}

## 1. Metadatos de cada parquet (esquema, filas, bbox) sin cargarlo entero

In [2]:
def parquet_meta(path: Path) -> dict:
    f = pq.ParquetFile(path)
    schema = f.schema_arrow
    cols = []
    for field in schema:
        if field.name in ("bbox",):          # columna auxiliar de GeoParquet (covering bbox)
            continue
        tipo = "geometry" if field.name == "geometry" else str(field.type)
        col = {"name": field.name, "type": tipo}
        if field.name in DESCRIPCIONES:
            col["description"] = DESCRIPCIONES[field.name]
        cols.append(col)
    meta = {"columns": cols, "rows": f.metadata.num_rows}
    geo = f.schema_arrow.metadata.get(b"geo") if f.schema_arrow.metadata else None
    if geo:
        import json
        g = json.loads(geo)
        meta["bbox"] = g["columns"]["geometry"]["bbox"]
        meta["geometry"] = "geometry"
    return meta


metas = {k: parquet_meta(p) for k, p in PRODUCTOS.items()}
for k, m in metas.items():
    print(f"{k:5s} filas={m['rows']:6d} cols={len(m['columns'])} bbox={m.get('bbox')}")

# La tabla puente no tiene geometría: hereda el bbox de la AGEB.
bbox_ageb = [round(v, 4) for v in metas["ageb"]["bbox"]]
bbox_mun = [round(v, 4) for v in metas["mun"]["bbox"]]
metas["grid"]["bbox"] = bbox_ageb
umbrales = pd.read_parquet(DERIVED / "clima_umbrales_2020.parquet").iloc[0].to_dict()

ent   filas=    32 cols=36 bbox=[-118.3651143522829, 14.532098369766466, -86.71040527988355, 32.71865357039013]
mun   filas=  2469 cols=49 bbox=[-118.3651143522829, 14.532098369766466, -86.71040527988355, 32.71865357039013]
ageb  filas= 79181 cols=59 bbox=[-118.3651143522829, 14.532098369766466, -86.71093514017757, 32.71865357039013]
grid  filas=248503 cols=27 bbox=None


## 2. Colección `inegi` dentro del catálogo existente

In [3]:
catalog = pystac.Catalog.from_file(str(STAC_DIR / "catalog.json"))
if catalog.get_child("inegi") is not None:
    catalog.remove_child("inegi")

# Censo 2020: levantamiento 2 – 27 de marzo, fecha de referencia 15 de marzo.
T0 = datetime(2020, 3, 2, tzinfo=timezone.utc)
T1 = datetime(2020, 3, 27, tzinfo=timezone.utc)
T_REF = datetime(2020, 3, 15, tzinfo=timezone.utc)

collection = pystac.Collection(
    id="inegi",
    title="INEGI — indicadores socioeconómicos (Censo 2020, MEDI)",
    description=(
        "Indicadores de carencia energética del diseño MEDI a partir del Censo de "
        "Población y Vivienda 2020 (INEGI): los siete componentes del MEDI "
        "(electricidad, refrigerador, teléfono, radio/TV del ITER; combustible, "
        "fogón sin chimenea y aire acondicionado del Cuestionario ampliado; "
        "calefacción de ENCEVI 2018) con confort térmico condicional al clima "
        "definido con la malla UTCI del atlas, y el índice MEDI (0–100) por "
        "estado, municipio y AGEB urbana sobre el Marco Geoestadístico 2020. "
        "Incluye la tabla puente AGEB → celda de 0.25° de la malla UTCI."
    ),
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([bbox_mun]),
        temporal=pystac.TemporalExtent([[T0, T1]]),
    ),
    license="other",
    keywords=["INEGI", "Censo 2020", "AGEB", "municipio", "MEDI", "pobreza energética", "México"],
    providers=[
        pystac.Provider(
            name="INEGI",
            description="Censo de Población y Vivienda 2020: Principales resultados por AGEB y manzana urbana; Marco Geoestadístico 2020",
            roles=[pystac.ProviderRole.PRODUCER, pystac.ProviderRole.LICENSOR],
            url="https://www.inegi.org.mx/programas/ccpv/2020/",
        ),
        pystac.Provider(
            name="IER-UNAM (atlas)",
            roles=[pystac.ProviderRole.PROCESSOR],
        ),
    ],
)
collection.add_link(pystac.Link(rel="license", target=LICENSE_URL, title="Términos de Libre Uso de la Información del INEGI"))
catalog.add_child(collection)

<Link rel=child target=<Collection id=inegi>>

## 3. Items (uno por producto) con extensión table y `atlas:indicators`

In [4]:
def bbox_geometry(b):
    return {"type": "Polygon", "coordinates": [[[b[0], b[1]], [b[2], b[1]], [b[2], b[3]], [b[0], b[3]], [b[0], b[1]]]]}


TITULOS = {
    "ent": "MEDI por entidad federativa, Censo 2020 (GeoParquet, EPSG:4326)",
    "ageb": "MEDI por AGEB urbana, Censo 2020 (GeoParquet, EPSG:4326)",
    "mun": "MEDI por municipio, Censo 2020 (GeoParquet, EPSG:4326)",
    "grid": "Tabla puente AGEB → celda UTCI 0.25° con conteos MEDI (Parquet)",
}
NIVEL = {"ent": "estado", "ageb": "ageb", "mun": "municipio", "grid": "ageb_grid"}

for k, path in PRODUCTOS.items():
    m = metas[k]
    props = {
        "start_datetime": T0.isoformat(),
        "end_datetime": T1.isoformat(),
        "atlas:level": NIVEL[k],
        "atlas:census_year": YEAR,
        "atlas:crs": "EPSG:4326" if k != "grid" else None,
        "atlas:grid_res_deg": 0.25 if k == "grid" else None,
        "atlas:indicators": indicators_for("ageb" if k == "grid" else k),
        "atlas:climate_rule": {k2: (float(v) if isinstance(v, (int, float)) else v) for k2, v in umbrales.items()},
        "atlas:source": [
            "INEGI. Censo de Población y Vivienda 2020. Principales resultados por AGEB y manzana urbana (datos abiertos, 32 archivos)",
            "INEGI. Marco Geoestadístico. Censo de Población y Vivienda 2020 (mg_2020_integrado)",
        ],
        "atlas:notebook": {"ent": "notebooks/010_MEDI_index.ipynb", "ageb": "notebooks/005_MEDI_ageb.ipynb + 010", "mun": "notebooks/006_MEDI_mun.ipynb + 010", "grid": "notebooks/006_MEDI_mun.ipynb + 010"}[k],
        "table:columns": m["columns"],
        "table:row_count": m["rows"],
    }
    if "geometry" in m:
        props["table:primary_geometry"] = "geometry"
    props = {a: b for a, b in props.items() if b is not None}

    item = pystac.Item(
        id=f"medi-{k}-{YEAR}",
        geometry=bbox_geometry(m["bbox"]),
        bbox=m["bbox"],
        datetime=T_REF,
        properties=props,
        stac_extensions=[TABLE_EXT],
    )
    item.add_asset(
        "data",
        pystac.Asset(
            href=str(path),
            media_type=PARQUET_MEDIA_TYPE,
            title=TITULOS[k],
            roles=["data"],
        ),
    )
    collection.add_item(item)
    print("item", item.id, "filas", m["rows"])

item medi-ent-2020 filas 32
item medi-mun-2020 filas 2469
item medi-ageb-2020 filas 79181
item medi-grid-2020 filas 248503


## 4. Guardar, validar y comprobar la ida y vuelta

In [5]:
catalog.normalize_hrefs(str(STAC_DIR))
catalog.make_all_asset_hrefs_relative()
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
print("Guardado en", STAC_DIR)
catalog.describe()

Guardado en /Users/gbv/atlas/data/stac
* <Catalog id=atlas>
    * <Collection id=utci>
      * <Item id=utci-hourly-2022>
      * <Item id=utci-levels-2022>
      * <Item id=utci-hourly-2023>
      * <Item id=utci-levels-2023>
    * <Collection id=inegi>
      * <Item id=medi-ent-2020>
      * <Item id=medi-mun-2020>
      * <Item id=medi-ageb-2020>
      * <Item id=medi-grid-2020>


In [6]:
for obj in [catalog, collection, *collection.get_items()]:
    obj.validate()
    print("OK", obj.id)
# La colección utci debe seguir intacta
utci = catalog.get_child("utci")
assert utci is not None and len(list(utci.get_items())) >= 2, "la colección utci se perdió"
print("utci intacta:", [it.id for it in utci.get_items()])

OK atlas
OK inegi


OK medi-ent-2020
OK medi-mun-2020
OK medi-ageb-2020
OK medi-grid-2020
utci intacta: ['utci-hourly-2022', 'utci-levels-2022', 'utci-hourly-2023', 'utci-levels-2023']


In [7]:
cat = pystac.Catalog.from_file(str(STAC_DIR / "catalog.json"))
col = cat.get_child("inegi")
for it in col.get_items():
    href = Path(it.assets["data"].get_absolute_href())
    assert href.exists(), href
    if "table:primary_geometry" in it.properties:
        df = gpd.read_parquet(href, columns=["cvegeo", "p_sin_refri", "geometry"])
        print(it.id, "→", href.relative_to(ROOT), "|", len(df), "filas, CRS", df.crs.to_epsg())
    else:
        df = pd.read_parquet(href)
        print(it.id, "→", href.relative_to(ROOT), "|", len(df), "filas (sin geometría)")
    assert len(df) == it.properties["table:row_count"]
for lv in ("ent", "mun", "ageb"):
    it = col.get_item(f"medi-{lv}-{YEAR}")
    print(lv, "indicadores:", [i["id"] for i in it.properties["atlas:indicators"]])
    cols_item = {c["name"] for c in it.properties["table:columns"]}
    faltan = [i["column"] for i in it.properties["atlas:indicators"] if i["column"] not in cols_item]
    assert not faltan, f"{lv}: columnas declaradas que no existen: {faltan}"

medi-ent-2020 → data/derived/INEGI/2020/medi_ent_2020.parquet | 32 filas, CRS 4326
medi-mun-2020 → data/derived/INEGI/2020/medi_mun_2020.parquet | 2469 filas, CRS 4326


medi-ageb-2020 → data/derived/INEGI/2020/medi_ageb_2020.parquet | 79181 filas, CRS 4326
medi-grid-2020 → data/derived/INEGI/2020/medi_ageb_2020_grid.parquet | 248503 filas (sin geometría)
ent indicadores: ['sin_elec', 'sin_refri', 'sin_confort', 'lena', 'fogon_sin_chim', 'sin_telef', 'sin_rtv', 'sin_ac', 'medi']
mun indicadores: ['sin_elec', 'sin_refri', 'sin_confort', 'lena', 'fogon_sin_chim', 'sin_telef', 'sin_rtv', 'sin_ac', 'medi']
ageb indicadores: ['sin_elec', 'sin_refri', 'sin_confort', 'lena', 'fogon_sin_chim', 'sin_telef', 'sin_rtv', 'sin_ac', 'medi']
